실험 A: 첫 번째 Conv2D(32)
실험 B: 첫 번째 Conv2D(64)

비교 항목:
1. val_accuracy
2. val_loss
3. total_params
4. test_accuracy
5. 관찰 결과

1. 실험 목표

비교할 모델은 두 개입니다.

실험 A: 첫 번째 Conv2D 필터 수 32개
실험 B: 첫 번째 Conv2D 필터 수 64개

기준 모델은 이겁니다.
Conv2D(32, kernel_size=3, activation='relu', padding='same')

바꿀 모델은 이겁니다.
Conv2D(64, kernel_size=3, activation='relu', padding='same')

즉, 첫 번째 Conv2D만 32 → 64로 바꿉니다.

2. 중요한 개념 먼저

기준 모델의 흐름은 이렇습니다.

입력
(28, 28, 1)

↓ Conv2D(32)

(28, 28, 32)

↓ MaxPooling2D

(14, 14, 32)

↓ Conv2D(64)

(14, 14, 64)

↓ MaxPooling2D

(7, 7, 64)

↓ Flatten

3136

↓ Dense(100)

↓ Dense(10)

그런데 첫 번째 Conv를 64로 바꾸면 흐름이 이렇게 됩니다.

입력
(28, 28, 1)

↓ Conv2D(64)

(28, 28, 64)

↓ MaxPooling2D

(14, 14, 64)

↓ Conv2D(64)

(14, 14, 64)

↓ MaxPooling2D

(7, 7, 64)

↓ Flatten

3136

↓ Dense(100)

↓ Dense(10)

여기서 핵심은 이겁니다.

첫 번째 Conv2D를 32에서 64로 바꾸면, 첫 번째 Conv의 출력 채널이 64가 됩니다. 그래서 두 번째 Conv2D의 입력 채널도 32가 아니라 64가 됩니다.

그래서 파라미터 수가 단순히 첫 번째 층에서만 늘어나는 게 아닙니다.
두 번째 Conv2D 파라미터도 같이 늘어납니다.

3. 파라미터 수 계산

(1) 실험 A: 첫 번째 Conv2D(32)
첫 번째 Conv2D
Conv2D(32, kernel_size=3)

입력 shape:
(28, 28, 1)

파라미터 공식:
(커널 높이 × 커널 너비 × 입력 채널 수 + bias 1개) × 필터 수

계산:
(3 × 3 × 1 + 1) × 32
= (9 + 1) × 32
= 10 × 32
= 320
첫 번째 Conv2D 파라미터 수는 320개입니다.

두 번째 Conv2D
Conv2D(64, kernel_size=3)

두 번째 Conv2D의 입력은 첫 번째 Pooling 결과입니다.

(14, 14, 32)

즉, 입력 채널 수는 32입니다.

계산:

(3 × 3 × 32 + 1) × 64
= (288 + 1) × 64
= 289 × 64
= 18,496

두 번째 Conv2D 파라미터 수는 18,496개입니다.

실험 B: 첫 번째 Conv2D(64)
첫 번째 Conv2D
Conv2D(64, kernel_size=3)

입력 채널은 여전히 1입니다.

계산:
(3 × 3 × 1 + 1) × 64
= (9 + 1) × 64
= 10 × 64
= 640
첫 번째 Conv2D 파라미터 수는 640개입니다.

두 번째 Conv2D
두 번째 Conv2D는 그대로 64 필터를 쓴다고 합시다.

Conv2D(64, kernel_size=3)
그런데 입력 채널이 달라졌습니다.

첫 번째 Conv가 64개 feature map을 만들었기 때문에, 첫 번째 Pooling 후 shape는:
(14, 14, 64)입니다.

따라서 두 번째 Conv2D의 입력 채널 수는 64입니다.

계산:
(3 × 3 × 64 + 1) × 64
= (576 + 1) × 64
= 577 × 64
= 36,928

두 번째 Conv2D 파라미터 수는 36,928개입니다.

입력채널의 변화를 자세히 살펴보겠습니다.

정확한 흐름은 이겁니다.

입력 이미지
(28, 28, 1)
= 28×28 흑백 이미지 1장

↓ 첫 번째 Conv2D(32)

(28, 28, 32)
= 28×28짜리 feature map 32장

↓ MaxPooling2D(2)

(14, 14, 32)
= 14×14짜리 feature map 32장

즉, 1장을 32장으로 바꾸는 것은 Pooling이 아니라 첫 번째 Conv2D(32)입니다.

1. 누가 1장을 32장으로 바꾸는가?

바로 이 코드입니다.

model.add(keras.layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'))

여기서 32는 필터 개수입니다.

Conv2D의 규칙은 이렇습니다.

필터 1개는 feature map 1장을 만든다.
필터 32개는 feature map 32장을 만든다.

그래서 입력이:

(28, 28, 1)

이면 첫 번째 Conv2D(32)를 지나서:

(28, 28, 32)

가 됩니다.

말로 풀면:

흑백 이미지 1장
↓
서로 다른 필터 32개가 각각 특징을 찾음
↓
특징 지도 32장 생성

2. Pooling은 무엇을 하냐?

Pooling은 장수를 바꾸지 않습니다.
Pooling은 각 feature map의 가로·세로 크기만 줄입니다.

model.add(keras.layers.MaxPooling2D(2))

이 코드는 2×2 영역에서 가장 큰 값 하나만 남깁니다.

그래서:

(28, 28, 32)
↓ MaxPooling2D(2)
(14, 14, 32)

가 됩니다.

변한 것:

28 → 14
28 → 14
32 → 32 그대로

즉:

28×28짜리 feature map 32장
↓
14×14짜리 feature map 32장

입니다.

3. 두 번째 Conv2D의 입력은 왜 32채널이냐?

두 번째 Conv2D 바로 앞의 출력이 이거였기 때문입니다.

(14, 14, 32)

이 말은:

14×14짜리 feature map이 32장 있다

는 뜻입니다.

그래서 두 번째 Conv2D는 원본 이미지 1장을 보는 게 아닙니다.
첫 번째 Conv2D가 만든 특징 지도 32장을 봅니다.

model.add(keras.layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'))

여기서 64는 두 번째 Conv2D가 새로 만들 feature map 수입니다.

입력: 14×14 feature map 32장
↓ Conv2D(64)
출력: 14×14 feature map 64장

4. 전체를 아주 쉽게 비유하면
원본 이미지
사진 1장
첫 번째 Conv2D(32)

사진을 32명의 분석가가 봅니다.

분석가 1: 선을 봄
분석가 2: 모서리를 봄
분석가 3: 곡선을 봄
...
분석가 32: 다른 작은 패턴을 봄

그래서 보고서가 32장 나옵니다.

특징 보고서 32장
MaxPooling

보고서 32장의 내용을 각각 요약합니다.

큰 보고서 32장
↓
작게 요약된 보고서 32장

보고서 장수는 그대로 32장입니다.
크기만 줄어듭니다.

두 번째 Conv2D(64)

이번에는 요약된 보고서 32장을 보고, 더 복잡한 관점의 보고서 64장을 새로 만듭니다.

요약된 특징 보고서 32장
↓
복합 특징 보고서 64장

5. 최종 정리
단계	            출력            shape	무슨 일이 일어남
입력	          (28, 28, 1) 	     흑백 이미지 1장
Conv2D(32)	     (28, 28, 32)	    필터 32개가 feature map 32장 생성
MaxPooling2D(2)	 (14, 14, 32)	    각 feature map의 크기만 절반으로 줄임
Conv2D(64)	     (14, 14, 64)	    32장 입력을 보고 64장 feature map 생성
MaxPooling2D(2)	 (7, 7, 64)	        64장의 크기만 절반으로 줄임

결론
Pooling은 한 장을 32장으로 바꾸지 않습니다.
한 장을 32장으로 바꾸는 것은 Conv2D(32)이고, Pooling은 그 32장의 가로·세로 크기만 줄입니다.

4. Conv 층 파라미터 비교
구분	첫 번째      Conv	두 번째 Conv	Conv 합계
실험 A: Conv2D(32)	320	    18,496	       18,816
실험 B: Conv2D(64)	640	    36,928	       37,568

정리하면:
첫 번째 Conv를 32 → 64로 바꾸면
첫 번째 Conv 파라미터: 320 → 640
두 번째 Conv 파라미터: 18,496 → 36,928
Conv 파라미터 합계: 18,816 → 37,568
즉, 거의 2배가 됩니다.

5. Dense 파라미터는 변하나?

이 구조에서는 Dense 파라미터는 그대로입니다.

왜냐하면 두 번째 Conv가 여전히 64개 필터를 출력하기 때문입니다.

두 모델 모두 두 번째 MaxPooling 후 shape는:

(7, 7, 64)

입니다.

Flatten 결과는 둘 다:

7 × 7 × 64 = 3136

입니다.

따라서 Dense(100) 파라미터도 둘 다 같습니다.

3136 × 100 + 100 = 313,700

마지막 Dense(10)도 같습니다.

100 × 10 + 10 = 1,010


6. 전체 파라미터 예상 비교
실험 A: 첫 번째 Conv2D(32)
Conv1: 320
Conv2: 18,496
Dense1: 313,700
Dense2: 1,010

총합:
320 + 18,496 + 313,700 + 1,010
= 333,526
실험 B: 첫 번째 Conv2D(64)
Conv1: 640
Conv2: 36,928
Dense1: 313,700
Dense2: 1,010

총합:
640 + 36,928 + 313,700 + 1,010
= 352,278

7. 전체 비교표
실험	첫 번째 Conv 필터 수	Conv1 params	Conv2 params	Dense params	Total params
A	    32	                    320	            18,496	        314,710	        333,526
B	    64	                    640	            36,928	        314,710	        352,278

여기서 Dense params는:

Dense(100):  313,700
Dense(10):   1,010
합계:         314,710 입니다.

In [ ]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# ============================================================
# 1. Fashion MNIST 데이터 불러오기
# ============================================================

(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

# 데이터 shape 확인
print("원본 train_input shape:", train_input.shape)
print("원본 test_input shape:", test_input.shape)


# ============================================================
# 2. CNN 입력 형태로 변환 + 스케일링
# ============================================================
# Conv2D는 입력을 4차원으로 받는다.
# 기본 형태: (batch_size, height, width, channels)
# Fashion MNIST는 흑백 이미지이므로 channels = 1

train_scaled = train_input.reshape(-1, 28, 28, 1) / 255.0
test_scaled = test_input.reshape(-1, 28, 28, 1) / 255.0

print("변환 후 train_scaled shape:", train_scaled.shape)
print("변환 후 test_scaled shape:", test_scaled.shape)


# ============================================================
# 3. 훈련 데이터 / 검증 데이터 분리
# ============================================================
# 전체 훈련 데이터 중 20%를 검증 데이터로 사용한다.

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled,
    train_target,
    test_size=0.2,
    random_state=42
)

print("학습용 train_scaled shape:", train_scaled.shape)
print("검증용 val_scaled shape:", val_scaled.shape)


# ============================================================
# 4. CNN 모델 생성 함수
# ============================================================
# first_filters 값만 바꿔서 Conv2D(32) 모델과 Conv2D(64) 모델을 만든다.

def make_cnn_model(first_filters=32):
    """
    first_filters:
    첫 번째 Conv2D 층의 필터 개수.
    
    first_filters=32 → 첫 번째 Conv2D(32)
    first_filters=64 → 첫 번째 Conv2D(64)
    """

    model = keras.Sequential()

    # 입력층
    # Fashion MNIST 한 장의 이미지 모양: 28 x 28 x 1
    model.add(keras.layers.Input(shape=(28, 28, 1)))

    # 첫 번째 Conv2D
    # 이 부분의 필터 수를 32 또는 64로 바꿔 실험한다.
    model.add(keras.layers.Conv2D(
        first_filters,
        kernel_size=3,
        activation='relu',
        padding='same'
    ))

    # 첫 번째 MaxPooling
    # 가로세로 크기를 절반으로 줄인다.
    model.add(keras.layers.MaxPooling2D(2))

    # 두 번째 Conv2D
    # 두 번째 Conv2D는 두 실험 모두 64개 필터로 고정한다.
    model.add(keras.layers.Conv2D(
        64,
        kernel_size=3,
        activation='relu',
        padding='same'
    ))

    # 두 번째 MaxPooling
    model.add(keras.layers.MaxPooling2D(2))

    # CNN 출력 feature map을 Dense에 넣기 위해 1차원으로 펼친다.
    model.add(keras.layers.Flatten())

    # Dense 은닉층
    model.add(keras.layers.Dense(100, activation='relu'))

    # 과적합 방지용 Dropout
    model.add(keras.layers.Dropout(0.4))

    # 출력층
    # Fashion MNIST는 10개 클래스 분류 문제이므로 출력 뉴런 10개
    model.add(keras.layers.Dense(10, activation='softmax'))

    return model


# ============================================================
# 5. 파라미터 계산 확인용 모델 summary
# ============================================================

print("\n" + "=" * 70)
print("실험 A: 첫 번째 Conv2D 필터 수 = 32")
print("=" * 70)

model_32 = make_cnn_model(first_filters=32)
model_32.summary()


print("\n" + "=" * 70)
print("실험 B: 첫 번째 Conv2D 필터 수 = 64")
print("=" * 70)

model_64 = make_cnn_model(first_filters=64)
model_64.summary()


# ============================================================
# 6. 실험 실행
# ============================================================

results = []

for first_filters in [32, 64]:
    print("\n" + "=" * 70)
    print(f"학습 시작: 첫 번째 Conv2D 필터 수 = {first_filters}")
    print("=" * 70)

    # 모델 생성
    model = make_cnn_model(first_filters=first_filters)

    # 모델 컴파일
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # 가장 좋은 모델 저장
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        f'best-cnn-first-conv-{first_filters}.keras',
        save_best_only=True
    )

    # 검증 손실이 좋아지지 않으면 조기 종료
    early_stopping_cb = keras.callbacks.EarlyStopping(
        patience=2,
        restore_best_weights=True
    )

    # 모델 학습
    history = model.fit(
        train_scaled,
        train_target,
        epochs=20,
        validation_data=(val_scaled, val_target),
        callbacks=[checkpoint_cb, early_stopping_cb],
        verbose=1
    )

    # 검증 데이터 평가
    val_loss, val_acc = model.evaluate(
        val_scaled,
        val_target,
        verbose=0
    )

    # 테스트 데이터 평가
    test_loss, test_acc = model.evaluate(
        test_scaled,
        test_target,
        verbose=0
    )

    # 전체 파라미터 수
    total_params = model.count_params()

    # 실험 결과 저장
    results.append({
        "first_conv_filters": first_filters,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "total_params": total_params,
        "epochs_trained": len(history.history["loss"])
    })


# ============================================================
# 7. 결과표 만들기
# ============================================================

result_df = pd.DataFrame(results)

print("\n실험 결과표")
display(result_df)


# ============================================================
# 8. 보기 좋게 퍼센트 컬럼 추가
# ============================================================

result_df["val_accuracy_percent"] = result_df["val_accuracy"] * 100
result_df["test_accuracy_percent"] = result_df["test_accuracy"] * 100

print("\n퍼센트 변환 결과표")
display(result_df)


# ============================================================
# 9. 검증 정확도 비교 그래프
# ============================================================

plt.bar(
    result_df["first_conv_filters"].astype(str),
    result_df["val_accuracy"]
)

plt.xlabel("First Conv2D filters")
plt.ylabel("Validation accuracy")
plt.title("Conv2D(32) vs Conv2D(64) - Validation Accuracy")
plt.ylim(0.85, 1.0)
plt.show()


# ============================================================
# 10. 테스트 정확도 비교 그래프
# ============================================================

plt.bar(
    result_df["first_conv_filters"].astype(str),
    result_df["test_accuracy"]
)

plt.xlabel("First Conv2D filters")
plt.ylabel("Test accuracy")
plt.title("Conv2D(32) vs Conv2D(64) - Test Accuracy")
plt.ylim(0.85, 1.0)
plt.show()


# ============================================================
# 11. 전체 파라미터 수 비교 그래프
# ============================================================

plt.bar(
    result_df["first_conv_filters"].astype(str),
    result_df["total_params"]
)

plt.xlabel("First Conv2D filters")
plt.ylabel("Total parameters")
plt.title("Conv2D(32) vs Conv2D(64) - Total Parameters")
plt.show()


# ============================================================
# 12. 제출용 관찰 문장 자동 출력
# ============================================================

params_32 = result_df.loc[result_df["first_conv_filters"] == 32, "total_params"].values[0]
params_64 = result_df.loc[result_df["first_conv_filters"] == 64, "total_params"].values[0]

val_acc_32 = result_df.loc[result_df["first_conv_filters"] == 32, "val_accuracy"].values[0]
val_acc_64 = result_df.loc[result_df["first_conv_filters"] == 64, "val_accuracy"].values[0]

print("\n제출용 관찰 문장")
print("-" * 70)

print(
    f"Conv2D(32) 모델의 총 파라미터 수는 {params_32:,}개이고, "
    f"Conv2D(64) 모델의 총 파라미터 수는 {params_64:,}개이다."
)

print(
    "첫 번째 Conv2D 필터 수를 32개에서 64개로 늘리면 "
    "첫 번째 Conv 층뿐 아니라 두 번째 Conv 층의 입력 채널 수도 증가하므로 "
    "전체 파라미터 수가 증가한다."
)

if val_acc_64 > val_acc_32:
    print(
        f"이번 실험에서는 Conv2D(64) 모델의 검증 정확도가 "
        f"{val_acc_64:.4f}로 Conv2D(32) 모델의 {val_acc_32:.4f}보다 높게 나타났다. "
        "다만 파라미터 수 증가가 항상 성능 향상을 보장하는 것은 아니다."
    )
elif val_acc_64 < val_acc_32:
    print(
        f"이번 실험에서는 Conv2D(64) 모델의 검증 정확도가 "
        f"{val_acc_64:.4f}로 Conv2D(32) 모델의 {val_acc_32:.4f}보다 낮게 나타났다. "
        "즉, 파라미터 수를 늘린다고 항상 성능이 좋아지는 것은 아니다."
    )
else:
    print(
        f"이번 실험에서는 두 모델의 검증 정확도가 모두 {val_acc_32:.4f}로 동일하게 나타났다. "
        "필터 수 증가가 성능 차이로 바로 이어지지는 않을 수 있다."
    )